# Tokens and Conversation Memory

Tokens are the atomic units of text that LLMs process. Every model has a context window (a hard limit on how many tokens it can receive in a single call). Because LLMs are stateless, simulating memory means passing the full conversation history in the messages list on every request. Understanding tokens is the prerequisite for every context management technique that follows — RAG, summarization, agents, and fine-tuning all depend on it.


In [1]:
import os
import gradio as gr
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

c:\Users\ksupr\Documents\llm_engineering\re-llm_engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# use OpenRouter key but OpenAI-compatible endpoint to demonstrate multi-provider setup
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
MODEL = "gpt-4o-mini"
CONTEXT_LIMIT = 128_000  # gpt-4o-mini context window

enc = tiktoken.encoding_for_model(MODEL)

### What is a Token?

A token is the atomic unit of text a model reads and writes.  
One token ≈ 4 characters or ¾ of a word in English.  
The tokenizer splits any string into token IDs before it reaches the model.

In [3]:
sample = "Transformers changed everything."

ids = enc.encode(sample)
print("Token IDs :", ids)
print("Decoded   :", enc.decode(ids))
print("Token count:", len(ids))
print()
print("Individual tokens:")
for id in ids:
    print(f"  {id:6d}  {repr(enc.decode([id]))}") 

Token IDs : [12200, 409, 9180, 5519, 13]
Decoded   : Transformers changed everything.
Token count: 5

Individual tokens:
   12200  'Transform'
     409  'ers'
    9180  ' changed'
    5519  ' everything'
      13  '.'


### Try different inputs

Notice how whitespace, punctuation, and casing affect token boundaries.

In [4]:
for text in [
    "hello world",
    "Hello World",
    "Hello, World!",
    "supercalifragilisticexpialidocious",
    "\u0393\u03b5\u03b9\u03b1 \u03c3\u03b1\u03c2",  # Greek: Geia sas
]:
    ids = enc.encode(text)
    print(f"{len(ids):3d} tokens | {repr(text)}")

  2 tokens | 'hello world'
  2 tokens | 'Hello World'
  4 tokens | 'Hello, World!'
 10 tokens | 'supercalifragilisticexpialidocious'
  3 tokens | 'Γεια σας'


### Counting Tokens in a Messages List

Before sending a request, count the tokens in the full messages list.  
This is how you track cost and avoid hitting the context window limit.

In [5]:
def count_tokens(messages: list[dict]) -> int:
    """Approximate token count for a messages list (content only)."""
    return sum(len(enc.encode(m["content"])) for m in messages)


def check_context(messages: list[dict]) -> None:
    """Print token usage and warn if approaching the context limit."""
    used = count_tokens(messages)
    pct = used / CONTEXT_LIMIT * 100
    print(f"Context: {used:,} / {CONTEXT_LIMIT:,} tokens ({pct:.1f}%)")
    if used > CONTEXT_LIMIT * 0.85:
        print("WARNING: approaching context limit — consider truncating history")

In [6]:
example_messages = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "What is a transformer model?"},
    {"role": "assistant", "content": "A transformer is a neural network architecture..."},
    {"role": "user",      "content": "How does self-attention work?"},
]

check_context(example_messages)

Context: 27 / 128,000 tokens (0.0%)


### LLM Statelessness and the Memory Pattern

LLMs have **no memory** between calls. Each call is independent.  
To simulate memory, append every turn (user + assistant) to the messages list and send the full history each time.

```bash
Call 1: [system, user₁]
Call 2: [system, user₁, assistant₁, user₂]
Call 3: [system, user₁, assistant₁, user₂, assistant₂, user₃]
```

The history *is* the memory.

In [7]:
def chat(history: list[dict], user_input: str) -> tuple[str, list[dict]]:
    """
    Append user turn, print token count, call API, append assistant reply.
    Returns reply and updated history.
    """
    history.append({"role": "user", "content": user_input})
    check_context(history)

    response = client.chat.completions.create(model=MODEL, messages=history)
    reply = response.choices[0].message.content

    history.append({"role": "assistant", "content": reply})
    return reply, history

### Multi-Turn Conversation Demo

Each turn appends to history. Watch the token count grow with each call.

In [8]:
history = [{"role": "system", "content": "You are a concise technical assistant."}]

turns = [
    "What is a token in the context of LLMs?",
    "Why does that matter for context windows?",
    "What happens when a conversation exceeds the context window?",
]

for turn in turns:
    print(f"\nUser: {turn}")
    reply, history = chat(history, turn)
    print(f"Assistant: {reply}")

print(f"\nTotal messages in history: {len(history)}")


User: What is a token in the context of LLMs?
Context: 19 / 128,000 tokens (0.0%)
Assistant: In the context of large language models (LLMs), a token is a unit of text that the model processes. Tokens can represent individual characters, words, or subwords, depending on the tokenization strategy used. For example:

- **Word-level tokenization** treats each word as a single token.
- **Subword tokenization**, used by models like BERT and GPT, breaks words into smaller components, allowing for more efficient handling of rare or complex words.

Tokens are the basic building blocks that LLMs use to understand and generate language, as models operate on these tokens during training and inference.

User: Why does that matter for context windows?
Context: 149 / 128,000 tokens (0.1%)
Assistant: Tokenization matters for context windows in LLMs because the number of tokens determines how much text can be processed at one time. The context window refers to the maximum number of tokens that the mod

### Token-Aware Chatbot

`gr.ChatInterface` manages the conversation UI.  
The token count and context usage are shown in the console on every turn.  
The system prompt is fixed; history is passed in full on each call.

In [9]:
SYSTEM_PROMPT = "You are a concise technical assistant."


def chat_ui(message: str, history: list[dict]) -> str:
    """
    Gradio ChatInterface callback.
    history is the list of prior {role, content} dicts managed by Gradio.
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [
        {"role": "user", "content": message}
    ]
    check_context(messages)  # prints token count to console

    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


demo = gr.ChatInterface(
    fn=chat_ui,
    type="messages",
    title="Token-Aware Chatbot",
    description="Multi-turn chat with live token count printed to console before each call.",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Token Inspector

Paste any text to see its token IDs, individual token strings, and total count.

In [10]:
def inspect_tokens(text: str) -> str:
    ids = enc.encode(text)
    lines = [f"**Total tokens: {len(ids)}**", ""]
    lines.append("| ID | Token |")
    lines.append("|-----|-------|")
    for id in ids:
        lines.append(f"| {id} | `{repr(enc.decode([id]))}` |")
    return "\n".join(lines)


inspector = gr.Interface(
    fn=inspect_tokens,
    inputs=gr.Textbox(lines=3, placeholder="Paste any text here...", label="Input"),
    outputs=gr.Markdown(label="Token breakdown"),
    title="Token Inspector",
    description="See how tiktoken splits your text into token IDs.",
    examples=[
        ["Transformers changed everything."],
        ["Hello, World! How are you?"],
        ["supercalifragilisticexpialidocious"],
    ]
)

inspector.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


### Context Window Limits

| Model | Context window |
|-------|---------------|
| gpt-4o-mini | 128,000 tokens |
| gpt-4o | 128,000 tokens |
| claude-3-5-sonnet | 200,000 tokens |
| gemini-1.5-pro | 1,000,000 tokens |

When the messages list exceeds the limit the API returns an error.  
Strategies to manage a full context window:

| Strategy | Trade-off |
|----------|-----------|
| Truncate oldest turns | Loses early context |
| Summarize history | Approximation; loses detail |
| Sliding window | Keeps only the last N turns |
| RAG | Retrieves relevant context on demand |

### Limitations

| Limitation | Impact |
|-----------|--------|
| Approximate token count | `sum(encode(content))` ignores per-message overhead — use for intuition, not billing |
| tiktoken is OpenAI-only | Other providers use different tokenizers; counts will differ |
| Growing history cost | Every turn adds to the prompt — long conversations become expensive |
| No automatic truncation | The API errors if you exceed the limit; you must manage history yourself |
| Statelessness is a feature | Full control over memory, but it is your responsibility to manage it |